<a href="https://colab.research.google.com/github/AntonioAgostini/C.V.-Final-Project-/blob/Giorgia's/CVFinalProject_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project 2: Joint Detection of AI-Generated Images and Post-Processing Alterations in Real-World Scenarios**

#**Imports**


In [ ]:
import os
import json
import random
import zipfile
import tarfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as transforms
from torchvision import models

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

from google.colab import drive

!pip install -q tqdm requests
import requests
from tqdm.auto import tqdm

#**Globals**

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ROOT_DIR = Path("/content")
DATA_DIR = ROOT_DIR / "RRDataset"
DOWNLOAD_DIR = ROOT_DIR / "downloads"
EXTRACT_DIR = ROOT_DIR / "RRDataset_extracted"
CHECKPOINT_DIR = ROOT_DIR / "checkpoints"

DOWNLOAD_DIR.mkdir(exist_ok=True)
EXTRACT_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 16
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4

RF_LABELS = {"real": 0, "fake": 1}
TRANS_LABELS = {
    "original": 0,
    "internet_transmitted": 1,
    "re_digitized": 2
}

Device: cuda
GPU: Tesla T4


# **Utils**

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def show_image(image_path):
    img = Image.open(image_path).convert("RGB")
    plt.figure(figsize=(4,4))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

def list_files_recursively(root_path, max_items=100):
    count = 0
    for root, dirs, files in os.walk(root_path):
        level = root.replace(str(root_path), "").count(os.sep)
        indent = " " * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = " " * 2 * (level + 1)
        for f in files:
            print(f"{subindent}{f}")
            count += 1
            if count >= max_items:
                print("\n[Output truncated]")
                return

In [ ]:
def download_with_progress(url, output_path, chunk_size=1024*1024):
    response = requests.get(url, stream=True)
    response.raise_for_status()

    total_size = int(response.headers.get("content-length", 0))

    with open(output_path, "wb") as f, tqdm(
        desc=output_path.name,
        total=total_size,
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for chunk in response.iter_content(chunk_size=chunk_size):
            if chunk:
                f.write(chunk)
                bar.update(len(chunk))

    print(f"Download completato: {output_path}")

#**Data**

### Download and Extract RRDataset
This cells downloads the RRDataset from Zenodo and extracts it into the local environment.

##Real file recovery from Zenodo

In [ ]:
import urllib.request

ZENODO_RECORD = "14963880"
ZENODO_API = f"https://zenodo.org/api/records/{ZENODO_RECORD}"

with urllib.request.urlopen(ZENODO_API) as response:
    record_meta = json.loads(response.read().decode())

files_in_record = record_meta.get("files", [])

print("File trovati nel record Zenodo:")
for i, f in enumerate(files_in_record):
    print(f"{i}: {f['key']}  |  {round(f.get('size', 0)/1e9, 2)} GB")

print("\nDettaglio completo dei file:\n")
for f in files_in_record:
    print("Nome file:", f["key"])
    print("Dimensione (GB):", round(f.get("size", 0)/1e9, 2))
    print("Link diretto:", f["links"]["self"])
    print("-" * 60)

File trovati nel record Zenodo:
0: RRDataset_original_train_val.tar.gz  |  2.16 GB
1: RRDataset_test.tar.gz  |  20.12 GB

Dettaglio completo dei file:

Nome file: RRDataset_original_train_val.tar.gz
Dimensione (GB): 2.16
Link diretto: https://zenodo.org/api/records/14963880/files/RRDataset_original_train_val.tar.gz/content
------------------------------------------------------------
Nome file: RRDataset_test.tar.gz
Dimensione (GB): 20.12
Link diretto: https://zenodo.org/api/records/14963880/files/RRDataset_test.tar.gz/content
------------------------------------------------------------


##Download Dataset

In [ ]:
target_file = files_in_record[0]   # poi eventualmente cambiamo
target_url = target_file["links"]["self"]
target_name = target_file["key"]

dest_path = DOWNLOAD_DIR / target_name

if not dest_path.exists():
    print("Scaricamento:", target_name)
    download_with_progress(target_url, dest_path)
else:
    print("File già presente:", dest_path)

File già presente: /content/downloads/RRDataset_original_train_val.tar.gz


##Archive extraction

In [ ]:
archive_path = dest_path

if str(archive_path).endswith(".zip"):
    with zipfile.ZipFile(archive_path, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print("Archivio ZIP estratto.")
elif str(archive_path).endswith(".tar.gz"):
    with tarfile.open(archive_path, "r:gz") as tf:
        tf.extractall(EXTRACT_DIR)
    print("Archivio TAR.GZ estratto.")
elif str(archive_path).endswith(".tar"):
    with tarfile.open(archive_path, "r:") as tf:
        tf.extractall(EXTRACT_DIR)
    print("Archivio TAR estratto.")
else:
    print("Formato non riconosciuto:", archive_path.name)

print("Prime cartelle trovate:")
for p in sorted(EXTRACT_DIR.rglob("*")):
    if p.is_dir():
        print(p)

/tmp/ipykernel_17849/3328273603.py:9: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(EXTRACT_DIR)


Archivio TAR.GZ estratto.
Prime cartelle trovate:
/content/RRDataset_extracted/RRDataset_original_train_val
/content/RRDataset_extracted/RRDataset_original_train_val/train
/content/RRDataset_extracted/RRDataset_original_train_val/train/ai
/content/RRDataset_extracted/RRDataset_original_train_val/train/real
/content/RRDataset_extracted/RRDataset_original_train_val/val
/content/RRDataset_extracted/RRDataset_original_train_val/val/ai
/content/RRDataset_extracted/RRDataset_original_train_val/val/real


##Dataset structure inspection

In [ ]:
#print("Struttura del dataset estratto:\n")
#list_files_recursively(EXTRACT_DIR, max_items=200)

##Image count

In [ ]:
IMG_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

all_images = []
for p in EXTRACT_DIR.rglob("*"):
    if p.suffix.lower() in IMG_EXTENSIONS:
        all_images.append(p)

print("Numero totale immagini trovate:", len(all_images))

#for p in all_images[:20]:
#    print(p)

Numero totale immagini trovate: 3000
/content/RRDataset_extracted/RRDataset_original_train_val/train/ai/Natural_Disasters_&_Accidents_000202.png
/content/RRDataset_extracted/RRDataset_original_train_val/train/ai/normal_009521.png
/content/RRDataset_extracted/RRDataset_original_train_val/train/ai/production_000936.png
/content/RRDataset_extracted/RRDataset_original_train_val/train/ai/War_&_Conflict_Scenes_000967.png
/content/RRDataset_extracted/RRDataset_original_train_val/train/ai/normal_009559.png
/content/RRDataset_extracted/RRDataset_original_train_val/train/ai/Medical_&_Public_Health_000051.png
/content/RRDataset_extracted/RRDataset_original_train_val/train/ai/Medical_&_Public_Health_000206.png
/content/RRDataset_extracted/RRDataset_original_train_val/train/ai/normal_009582.png
/content/RRDataset_extracted/RRDataset_original_train_val/train/ai/normal_007943.png
/content/RRDataset_extracted/RRDataset_original_train_val/train/ai/War_&_Conflict_Scenes_000799.png
/content/RRDataset_ext

#**Network**

# **Train**

#**Evaluation/Test**